# URL-Based Keyword and Entity Extraction using IBM WatsonX.ai

This notebook demonstrates how to:
1. **Extract content from URLs** using web scraping
2. **Extract keywords, phrases, and concepts** using IBM WatsonX.ai
3. **Extract named entities** (PERSON, ORGANIZATION, LOCATION, DATE, EVENT, PRODUCT, OTHER)
4. **Classify content into categories**
5. **Handle errors gracefully** - If web scraping fails, skip keyword/entity extraction

## Table of Contents
1. [Setup and Installation](#setup)
2. [Initialize Web Scraper](#web-scraper)
3. [Initialize WatsonX.ai Client](#init-watsonx)
4. [Initialize Classification Pipeline](#init-pipeline)
5. [Single URL Analysis](#single-url)
6. [Batch URL Processing](#batch-processing)
7. [Save Results](#save-results)
8. [Statistical Analysis](#statistical-analysis)

## Key Features
- ✅ **Web Scraping**: Automatic content extraction from URLs
- ✅ **Error Handling**: Skips extraction if scraping fails
- ✅ **Dual Text Output**: Raw and cleaned text
- ✅ **Keyword Extraction**: Theme, keywords, phrases, concepts
- ✅ **Entity Recognition**: 7 entity types
- ✅ **Category Classification**: Multi-label classification
- ✅ **Batch Processing**: Process multiple URLs from CSV
- ✅ **Complete Export**: CSV and JSON with all fields

## 1. Setup and Installation <a name="setup"></a>

First, install the required packages if not already installed.

In [ ]:
# Uncomment to install required packages
# !pip install requests beautifulsoup4 lxml ibm-watsonx-ai sentence-transformers pandas numpy matplotlib seaborn python-dotenv

### Import Required Libraries

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import json
import re
from typing import List, Dict, Optional, Any
from collections import Counter
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns

# Add app directory to path for imports
sys.path.append('../app')

# Import our custom modules
from src.pipeline import ClassificationPipeline

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ All libraries imported successfully")

/Users/abhilashamangal/miniconda3/envs/elastic/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All libraries imported successfully


### Configure Credentials

Set up your IBM Cloud credentials. Refer to `IBM_CREDENTIALS_SETUP.md` for detailed instructions.

In [2]:
# Load environment variables
load_dotenv()

WATSONX_API_KEY = os.getenv("WATSONX_API_KEY")
WATSONX_PROJECT_ID = os.getenv("WATSONX_PROJECT_ID")
DATA_PATH = os.getenv("DATA_PATH", "../data/with_label.csv")

if not WATSONX_API_KEY or not WATSONX_PROJECT_ID:
    raise ValueError("❌ Please set WATSONX_API_KEY and WATSONX_PROJECT_ID in .env file")

print("✓ Credentials loaded successfully")
print(f"✓ Data path: {DATA_PATH}")

✓ Credentials loaded successfully
✓ Data path: /Users/abhilashamangal/Documents/2026/Github/ShareThis-Predictive-Multilabel-Classification/data/with_label.csv


## 2. Initialize Classification Pipeline <a name="init-pipeline"></a>

The pipeline includes:
- Web scraper for URL content extraction
- Category classifier
- Keyword extractor
- Entity extractor

In [4]:
# Initialize pipeline
print("Initializing Classification Pipeline...")
pipeline = ClassificationPipeline(
    watsonx_api_key=WATSONX_API_KEY,
    watsonx_project_id=WATSONX_PROJECT_ID,
    data_path=DATA_PATH,
    use_cos=False
)

print("✓ Pipeline initialized")

2026-04-24 16:45:10,631 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Initializing Classification Pipeline...


2026-04-24 16:45:13,465 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: mps
2026-04-24 16:45:14,722 - ibm_watsonx_ai.client - INFO - Client successfully initialized
2026-04-24 16:45:15,830 - httpx - INFO - HTTP Request: GET https://us-south.ml.cloud.ibm.com/ml/v1/foundation_model_specs?version=2025-04-23&project_id=b1265848-fa75-415a-887b-2fca8c310f6c&filters=function_text_generation%2C%21lifecycle_withdrawn%3Aand&limit=200 "HTTP/1.1 200 OK"
2026-04-24 16:45:16,599 - ibm_watsonx_ai.wml_resource - INFO - Successfully finished Get available foundation models for url: 'https://us-south.ml.cloud.ibm.com/ml/v1/foundation_model_specs?version=2025-04-23&project_id=b1265848-fa75-415a-887b-2fca8c310f6c&filters=function_text_generation%2C%21lifecycle_withdrawn%3Aand&limit=200'


✓ Pipeline initialized


In [5]:
# Load and prepare data
print("Loading and preparing data...")
pipeline.load_and_prepare_data()
print(f"✓ Loaded {len(pipeline.df)} records")
print(f"✓ Found {len(pipeline.unique_categories)} unique categories")

Loading and preparing data...
✓ Loaded 1060 records
✓ Found 574 unique categories


In [6]:
# Generate embeddings
print("Generating embeddings...")
pipeline.generate_embeddings()
print("✓ Embeddings generated")

Generating embeddings...


Batches: 100%|██████████| 34/34 [00:03<00:00,  9.84it/s]

✓ Embeddings generated


In [7]:
# Create frequency buckets
print("Creating frequency buckets...")
pipeline.create_frequency_buckets()
print("✓ Frequency buckets created")

Creating frequency buckets...
✓ Frequency buckets created


In [8]:
# Prepare examples
print("Preparing examples...")
pipeline.prepare_examples(n_samples=10)
print("✓ Examples prepared")

Preparing examples...
✓ Examples prepared


In [9]:
# Initialize keyword and entity extractor
print("Initializing keyword and entity extractor...")
pipeline.initialize_extractor()
print("✓ Extractor initialized")
print("\n🎉 Pipeline is ready for URL analysis!")

Initializing keyword and entity extractor...
✓ Extractor initialized

🎉 Pipeline is ready for URL analysis!


## 3. Single URL Analysis <a name="single-url"></a>

Let's analyze a single URL to see the complete pipeline in action.

### Process Flow:
1. **Web Scraping**: Extract content from URL
2. **Error Check**: If scraping fails, return error (skip extraction)
3. **Classification**: Classify content into categories
4. **Keyword Extraction**: Extract theme, keywords, phrases, concepts
5. **Entity Extraction**: Extract named entities (7 types)
6. **Return Results**: All outputs with both raw and cleaned text

In [ ]:
# Example URL to analyze
test_url = "https://techcrunch.com/2024/01/15/openai-announces-gpt-4-turbo/"

print(f"Analyzing URL: {test_url}")
print("\nProcess:")
print("  1. Scraping webpage...")
print("  2. Extracting and cleaning text...")
print("  3. Classifying into categories...")
print("  4. Extracting keywords and phrases...")
print("  5. Extracting named entities...")
print("\nPlease wait...\n")

result = pipeline.analyze_url(
    url=test_url,
    k=55,
    max_keywords=10
)

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

### Display Results

**Note**: If web scraping fails, the result will contain an error message and empty arrays for keywords/entities.

In [ ]:
# Display results
print(f"\n📊 RESULTS FOR: {result['url']}")
print(f"\nStatus: {result['status']}")

if result['status'] == 'success':
    print("\n" + "="*80)
    print("📁 METADATA")
    print("="*80)
    print(f"Title: {result['metadata'].get('title', 'N/A')}")
    print(f"Description: {result['metadata'].get('description', 'N/A')}")
    print(f"Domain: {result['metadata'].get('domain', 'N/A')}")
    print(f"Word Count: {result['metadata'].get('word_count', 'N/A')}")
    
    print("\n" + "="*80)
    print("🏷️  CATEGORIES")
    print("="*80)
    if result['categories']:
        for i, cat in enumerate(result['categories'], 1):
            print(f"{i}. {cat}")
    else:
        print("No categories found")
    
    print("\n" + "="*80)
    print("🔑 KEYWORDS")
    print("="*80)
    
    # Check extraction status
    if result['keywords'].get('extraction_status') == 'success':
        print(f"\nTheme: {result['keywords']['theme']}")
        print(f"\nKeywords: {', '.join(result['keywords']['keywords']) if result['keywords']['keywords'] else 'None'}")
        print(f"\nPhrases: {', '.join(result['keywords']['phrases']) if result['keywords']['phrases'] else 'None'}")
        print(f"\nConcepts: {', '.join(result['keywords']['concepts']) if result['keywords']['concepts'] else 'None'}")
    else:
        print(f"\n⚠️  Keyword extraction failed: {result['keywords'].get('error', 'Unknown error')}")
        if result['keywords'].get('raw_response'):
            print(f"\nRaw response (first 200 chars): {result['keywords']['raw_response'][:200]}...")
    
    print("\n" + "="*80)
    print("👤 ENTITIES")
    print("="*80)
    
    # Check extraction status
    if result['entities'].get('extraction_status') == 'success':
        for entity_type in ['PERSON', 'ORGANIZATION', 'LOCATION', 'DATE', 'EVENT', 'PRODUCT', 'OTHER']:
            entities = result['entities'].get(entity_type, [])
            if entities:
                print(f"\n{entity_type}: {', '.join(entities)}")
        
        # Check if no entities found
        total_entities = sum(len(result['entities'].get(t, [])) for t in ['PERSON', 'ORGANIZATION', 'LOCATION', 'DATE', 'EVENT', 'PRODUCT', 'OTHER'])
        if total_entities == 0:
            print("\nNo entities found in the text")
    else:
        print(f"\n⚠️  Entity extraction failed: {result['entities'].get('error', 'Unknown error')}")
        if result['entities'].get('raw_response'):
            print(f"\nRaw response (first 200 chars): {result['entities']['raw_response'][:200]}...")
    
    print("\n" + "="*80)
    print("📄 EXTRACTED TEXT")
    print("="*80)
    if result.get('raw_text'):
        print(f"\nRaw Text (first 500 chars):\n{result['raw_text'][:500]}...")
    if result.get('cleaned_text'):
        print(f"\nCleaned Text (first 500 chars):\n{result['cleaned_text'][:500]}...")
else:
    print(f"\n❌ Error: {result.get('error', 'Unknown error')}")
    print("\n⚠️  Since web scraping failed, keyword and entity extraction were skipped.")

## 4. Batch URL Processing <a name="batch-processing"></a>

Process multiple URLs from a CSV file.

### Error Handling:
- If web scraping fails for a URL, the result will have `status='error'`
- Keywords and entities will be empty arrays
- The error message will be included in the `error` field
- Processing continues with the next URL

### Create Sample CSV (Optional)

If you don't have a CSV file, run this cell to create a sample one:

In [ ]:
# Create sample CSV with URLs
sample_urls = [
    "https://techcrunch.com/2024/01/15/ai-breakthrough/",
    "https://www.theverge.com/2024/1/10/tech-news/",
    "https://www.wired.com/story/future-of-ai/",
    "https://arstechnica.com/gadgets/2024/01/new-smartphone/",
    "https://www.bbc.com/news/technology-12345678",
    "https://invalid-url-that-will-fail.com/test"  # This will fail
]

sample_df = pd.DataFrame({'url': sample_urls})
sample_df.to_csv('../data/sample_urls_for_extraction.csv', index=False)
print("✓ Sample CSV created: ../data/sample_urls_for_extraction.csv")
print(f"\nSample URLs ({len(sample_urls)} total):")
for i, url in enumerate(sample_urls, 1):
    print(f"{i}. {url}")

### Load URLs from CSV

In [ ]:
# Load CSV file with URLs
# Change this path to your CSV file
csv_file_path = '../data/sample_urls_for_extraction.csv'

# Read CSV
urls_df = pd.read_csv(csv_file_path)

print(f"✓ Loaded {len(urls_df)} URLs from CSV")
print(f"\nColumns: {list(urls_df.columns)}")
print(f"\nFirst 5 URLs:")
print(urls_df.head())

### Process All URLs

**Important**: 
- If web scraping fails, the URL will be marked as `status='error'`
- Keywords and entities will be empty
- Processing continues with remaining URLs

In [ ]:
# Process all URLs
results = []
total_urls = len(urls_df)

print(f"Processing {total_urls} URLs...\n")
print("="*80)

for idx, row in urls_df.iterrows():
    url = row['url']
    print(f"\n[{idx + 1}/{total_urls}] Processing: {url}")
    
    try:
        result = pipeline.analyze_url(
            url=url,
            k=55,
            max_keywords=10
        )
        results.append(result)
        
        if result['status'] == 'success':
            print(f"  ✓ Success")
            print(f"  Categories: {len(result['categories'])}")
            
            # Check keyword extraction status
            if result['keywords'].get('extraction_status') == 'success':
                print(f"  Keywords: {len(result['keywords']['keywords'])}")
            else:
                print(f"  Keywords: Failed - {result['keywords'].get('error', 'Unknown')}")
            
            # Check entity extraction status
            if result['entities'].get('extraction_status') == 'success':
                total_entities = sum(len(v) for k, v in result['entities'].items() if k != 'extraction_status')
                print(f"  Entities: {total_entities}")
            else:
                print(f"  Entities: Failed - {result['entities'].get('error', 'Unknown')}")
        else:
            print(f"  ✗ Failed: {result.get('error', 'Unknown error')}")
            print(f"  ⚠️  Keyword and entity extraction skipped due to scraping failure")
    
    except Exception as e:
        print(f"  ✗ Exception: {str(e)}")
        results.append({
            'url': url,
            'status': 'error',
            'error': str(e),
            'categories': [],
            'keywords': {'extraction_status': 'skipped'},
            'entities': {'extraction_status': 'skipped'},
            'metadata': {},
            'raw_text': None,
            'cleaned_text': None
        })

print("\n" + "="*80)
print(f"\n✓ Processed {len(results)} URLs")
successful = sum(1 for r in results if r['status'] == 'success')
failed = len(results) - successful
print(f"  Success: {successful}")
print(f"  Failed: {failed}")

## 5. Results Analysis <a name="statistical-analysis"></a>

In [ ]:
# Convert results to DataFrame for easier analysis
results_data = []

for result in results:
    # Extract keyword extraction status
    keyword_status = result['keywords'].get('extraction_status', 'unknown')
    entity_status = result['entities'].get('extraction_status', 'unknown')
    
    row = {
        'url': result['url'],
        'status': result['status'],
        'title': result['metadata'].get('title', ''),
        'domain': result['metadata'].get('domain', ''),
        'word_count': result['metadata'].get('word_count', 0),
        'categories': ', '.join(result['categories']),
        'num_categories': len(result['categories']),
        'keyword_extraction_status': keyword_status,
        'entity_extraction_status': entity_status,
        'theme': result['keywords'].get('theme', ''),
        'keywords': ', '.join(result['keywords'].get('keywords', [])),
        'num_keywords': len(result['keywords'].get('keywords', [])),
        'phrases': ', '.join(result['keywords'].get('phrases', [])),
        'num_phrases': len(result['keywords'].get('phrases', [])),
        'concepts': ', '.join(result['keywords'].get('concepts', [])),
        'num_concepts': len(result['keywords'].get('concepts', [])),
        'persons': ', '.join(result['entities'].get('PERSON', [])),
        'organizations': ', '.join(result['entities'].get('ORGANIZATION', [])),
        'locations': ', '.join(result['entities'].get('LOCATION', [])),
        'dates': ', '.join(result['entities'].get('DATE', [])),
        'events': ', '.join(result['entities'].get('EVENT', [])),
        'products': ', '.join(result['entities'].get('PRODUCT', [])),
        'other_entities': ', '.join(result['entities'].get('OTHER', [])),
        'total_entities': sum(len(v) for k, v in result['entities'].items() if k not in ['extraction_status', 'error', 'raw_response']),
        'raw_text': result.get('raw_text', ''),
        'cleaned_text': result.get('cleaned_text', ''),
        'error': result.get('error', '')
    }
    results_data.append(row)

results_df = pd.DataFrame(results_data)
print("✓ Results converted to DataFrame")
print(f"\nDataFrame shape: {results_df.shape}")
print(f"Columns: {list(results_df.columns)}")

In [ ]:
# Display summary statistics
print("\n" + "="*80)
print("📊 SUMMARY STATISTICS")
print("="*80)

print(f"\nTotal URLs processed: {len(results_df)}")
print(f"Successful: {len(results_df[results_df['status'] == 'success'])}")
print(f"Failed: {len(results_df[results_df['status'] == 'error'])}")

successful_df = results_df[results_df['status'] == 'success']

if len(successful_df) > 0:
    print(f"\n--- Successful Extractions ---")
    print(f"Average word count: {successful_df['word_count'].mean():.0f}")
    print(f"Average categories per URL: {successful_df['num_categories'].mean():.1f}")
    
    # Keyword extraction stats
    keyword_success = len(successful_df[successful_df['keyword_extraction_status'] == 'success'])
    print(f"\nKeyword extraction success rate: {keyword_success}/{len(successful_df)} ({keyword_success/len(successful_df)*100:.1f}%)")
    if keyword_success > 0:
        keyword_df = successful_df[successful_df['keyword_extraction_status'] == 'success']
        print(f"Average keywords per URL: {keyword_df['num_keywords'].mean():.1f}")
        print(f"Average phrases per URL: {keyword_df['num_phrases'].mean():.1f}")
        print(f"Average concepts per URL: {keyword_df['num_concepts'].mean():.1f}")
    
    # Entity extraction stats
    entity_success = len(successful_df[successful_df['entity_extraction_status'] == 'success'])
    print(f"\nEntity extraction success rate: {entity_success}/{len(successful_df)} ({entity_success/len(successful_df)*100:.1f}%)")
    if entity_success > 0:
        entity_df = successful_df[successful_df['entity_extraction_status'] == 'success']
        print(f"Average entities per URL: {entity_df['total_entities'].mean():.1f}")

In [ ]:
# Display first few results
print("\n" + "="*80)
print("📋 SAMPLE RESULTS")
print("="*80)

display_columns = ['url', 'status', 'title', 'num_categories', 'keyword_extraction_status', 'entity_extraction_status', 'num_keywords', 'total_entities']
print(results_df[display_columns].head(10))

## 6. Save Results <a name="save-results"></a>

Export results to CSV and JSON formats.

In [ ]:
# Export to CSV
output_csv = '../data/url_keyword_entity_extraction_results.csv'
results_df.to_csv(output_csv, index=False)
print(f"✓ Results exported to CSV: {output_csv}")
print(f"  Columns: {len(results_df.columns)}")
print(f"  Rows: {len(results_df)}")

In [ ]:
# Export to JSON (with full structure)
output_json = '../data/url_keyword_entity_extraction_results.json'
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"✓ Full results exported to JSON: {output_json}")
print(f"  Total records: {len(results)}")

## 7. Additional Analysis <a name="additional-analysis"></a>

### Filter by Extraction Status

In [ ]:
# Show URLs where keyword extraction failed
failed_keywords = results_df[results_df['keyword_extraction_status'] != 'success']
if len(failed_keywords) > 0:
    print(f"\n⚠️  URLs with failed keyword extraction: {len(failed_keywords)}")
    print(failed_keywords[['url', 'status', 'keyword_extraction_status', 'error']].head())
else:
    print("\n✓ All keyword extractions successful!")

In [ ]:
# Show URLs where entity extraction failed
failed_entities = results_df[results_df['entity_extraction_status'] != 'success']
if len(failed_entities) > 0:
    print(f"\n⚠️  URLs with failed entity extraction: {len(failed_entities)}")
    print(failed_entities[['url', 'status', 'entity_extraction_status', 'error']].head())
else:
    print("\n✓ All entity extractions successful!")

### Find Most Common Keywords and Entities

In [ ]:
# Collect all keywords from successful extractions
all_keywords = []
for result in results:
    if result['status'] == 'success' and result['keywords'].get('extraction_status') == 'success':
        all_keywords.extend(result['keywords'].get('keywords', []))

if all_keywords:
    keyword_counts = Counter(all_keywords)
    print("\nTop 10 Most Common Keywords:")
    for keyword, count in keyword_counts.most_common(10):
        print(f"  {keyword}: {count}")
else:
    print("\nNo keywords extracted")

In [ ]:
# Collect all organizations from successful extractions
all_orgs = []
for result in results:
    if result['status'] == 'success' and result['entities'].get('extraction_status') == 'success':
        all_orgs.extend(result['entities'].get('ORGANIZATION', []))

if all_orgs:
    org_counts = Counter(all_orgs)
    print("\nTop 10 Most Mentioned Organizations:")
    for org, count in org_counts.most_common(10):
        print(f"  {org}: {count}")
else:
    print("\nNo organizations extracted")

## 🎉 Analysis Complete!

You have successfully:
1. ✅ Initialized the classification pipeline with web scraper
2. ✅ Analyzed URLs with automatic content extraction
3. ✅ Extracted categories, keywords, and entities
4. ✅ Handled errors gracefully (skipped extraction when scraping failed)
5. ✅ Processed batch URLs from CSV
6. ✅ Exported results to CSV and JSON
7. ✅ Analyzed extraction success rates

### Key Features:
- **Error Handling**: If web scraping fails, keyword/entity extraction is skipped
- **Status Tracking**: Each extraction has a status (success/failed/error/skipped)
- **Detailed Logging**: Check console for detailed extraction logs
- **Complete Export**: All fields including texts and errors in CSV/JSON

### Next Steps:
- Review the exported CSV file for tabular results
- Check the JSON file for complete structured data
- Analyze extraction success rates
- Investigate failed extractions using error messages
- Use the results for further analysis or visualization

---

**Created by**: Bob  
**Date**: 2026-04-24  
**Version**: 2.0.0 (Updated with web scraping and error handling)